In [1]:
import sys
sys.path.insert(0, '/home/joseluisalmendarezgonzalez/Desktop/3_GEO_FNO')

In [2]:
import torch
import random
import numpy as np
from torch.utils.data import DataLoader, random_split
from lib.Common import setup_logging, KolmogorovDataset, FNOGenerator, NavierStokesResiduo
from lib.GenAdvNetworkApproach import FNODiscriminatorStat, FNODiscriminatorPhys, WGAFNOGPTrainer

In [4]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [5]:
setup_logging("gan_experiment.log")

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH = "../Dataset/snapshots_64x64_use.npy"

In [8]:
dataset = KolmogorovDataset(DATA_PATH, seq_len=10)
train_len = int(0.8 * len(dataset))
val_len = len(dataset) - train_len
train_ds, val_ds = random_split(dataset, [train_len, val_len])

2026-06-07 19:31:30,313 | INFO | Dataset: 1152 trayectorias × 90 ventanas = 103,680 muestras | seq_len=10 | H×W=64×64


In [9]:
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=4)

In [10]:
G = FNOGenerator(hidden_ch=32, modes1=12, modes2=12, n_layers=4, z_dim=8).to(DEVICE)
D_stat = FNODiscriminatorStat(seq_len=11, hidden_ch=32, modes1=12, modes2=12, n_layers=4).to(DEVICE)
D_phys = FNODiscriminatorPhys(seq_len=11, hidden_ch=32, modes1=12, modes2=12, n_layers=4).to(DEVICE)
ns = NavierStokesResiduo(64, 64, dt=0.05, modes1=12, modes2=12, device=DEVICE).to(DEVICE)

In [11]:
trainer = WGAFNOGPTrainer(
    G, D_stat, D_phys, ns, DEVICE,
    n_critic=2, lr_G=1e-4, lr_D=1e-4,
    log_dir="logs_gan",
    resume=True,          # ← reanuda automáticamente si existe checkpoint
    vis_freq=1,
)

2026-06-07 19:31:44,034 | WARNING | No se encontró checkpoint para reanudar; empezando desde cero.


In [12]:
history = trainer.fit(train_loader, val_loader, n_epochs=30, log_every=1)
print(history)

Epoch    0/30:   0%|                                                     | 0/10368 [00:00<?, ?it/s]/home/joseluisalmendarezgonzalez/miniconda3/envs/py_env/lib/python3.10/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,
                                                                                                   

KeyboardInterrupt: 